# Case 02 · LoRA from scratch

**Goal:** implement LoRA yourself in ~30 lines, prove it matches full fine-tuning at a fraction of the
trainable parameters, and understand *why* it works. This is the engine behind `peft` and QLoRA (Case 03).

Pair with `animation.html`. Runs on CPU.

---
### The idea in one line
> Freeze the pretrained weight **W**. Learn a low-rank update **ΔW = B·A·(α/r)** and use **W + ΔW**.
> Only the skinny matrices **A** (r×in) and **B** (out×r) are trained.

Why it's allowed: empirically, the *change* a task needs is low-rank — it lives in a few directions,
so two thin matrices can represent it.

In [1]:
import torch, torch.nn as nn, math, copy
import matplotlib.pyplot as plt
torch.manual_seed(0)

## 1 · The LoRA layer (the whole trick)

Wrap a frozen `nn.Linear` with a trainable low-rank side-path. Note **B is initialized to zero** so the
adapted model *starts identical* to the pretrained one — a safe, no-surprise start.

In [2]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=4, alpha=8):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad_(False)   # FREEZE W
        in_f, out_f = base.in_features, base.out_features
        self.scale = alpha / r
        self.A = nn.Parameter(torch.randn(r, in_f) * 0.01)          # small random
        self.B = nn.Parameter(torch.zeros(out_f, r))                # zero -> ΔW=0 at start
    def forward(self, x):
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.scale

The `LoRALinear` class is a custom PyTorch module designed to implement the LoRA (Low-Rank Adaptation) technique for linear layers. Let's break down its components:

*   **`__init__(self, base: nn.Linear, r=4, alpha=8)`**: This is the constructor for the `LoRALinear` module.
    *   `base: nn.Linear`: It takes an existing `torch.nn.Linear` layer as its `base`. This is the pre-trained linear layer that we want to adapt.
    *   `r` (rank): This parameter determines the rank of the low-rank matrices `A` and `B`. A smaller `r` means fewer trainable parameters. The default is 4.
    *   `alpha`: This is a scaling factor for the LoRA update. The default is 8.
    *   `self.base = base`: Stores the original linear layer.
    *   `for p in self.base.parameters(): p.requires_grad_(False)`: **Crucially, this line freezes the weights of the `base` linear layer**, meaning they will not be updated during training. This is a core idea of LoRA.
    *   `in_f, out_f = base.in_features, base.out_features`: Retrieves the input and output feature dimensions from the base layer.
    *   `self.scale = alpha / r`: Calculates the scaling factor that will be applied to the low-rank update.
    *   `self.A = nn.Parameter(torch.randn(r, in_f) * 0.01)`: Initializes `A` as a `torch.nn.Parameter`. It's a matrix of shape `(r, in_f)` initialized with small random values. This matrix will be trained.
    *   `self.B = nn.Parameter(torch.zeros(out_f, r))`: Initializes `B` as a `torch.nn.Parameter`. It's a matrix of shape `(out_f, r)` initialized with zeros. This initialization ensures that `B @ A` (and thus the LoRA update `ΔW`) is initially zero, so the adapted model starts identical to the pre-trained one.

*   **`forward(self, x)`**: This method defines how the `LoRALinear` module processes its input `x`.
    *   `self.base(x)`: This is the output of the original, frozen linear layer.
    *   `(x @ self.A.t() @ self.B.t())`: This calculates the low-rank update. The input `x` is multiplied by the transpose of `A` (`A.t()`) and then by the transpose of `B` (`B.t()`). This effectively computes `x @ ΔW`, where `ΔW = B @ A`. The `t()` is used because `torch.nn.Linear`'s weight matrix is typically `(out_features, in_features)`, while LoRA's `ΔW = B @ A` is `(out_features, in_features)` (with `A` being `(r, in_features)` and `B` being `(out_features, r)`). The matrix multiplication `x @ A.t() @ B.t()` correctly implements `x @ (B @ A).t()` when `x` is `(batch_size, in_features)`.
    *   `* self.scale`: The calculated low-rank update is then scaled by `alpha / r`.
    *   `return self.base(x) + (...)`: The final output is the sum of the original frozen base layer's output and the scaled low-rank update. This means the effective weight matrix used is `W_base + (B @ A) * (alpha / r)`.

In essence, `LoRALinear` allows you to adapt a pre-trained linear layer by only training two much smaller matrices (`A` and `B`) instead of the entire original weight matrix, significantly reducing the number of trainable parameters while achieving comparable performance.

## What are low-rank matrices A and B?

The term "low-rank matrices A and B" refers to a core concept in the LoRA technique, which is designed to efficiently update neural network weights.

Let's break down what a **low-rank matrix** means:

*   **Rank of a Matrix:** In linear algebra, the rank of a matrix is a measure of its "degeneracy" or the number of linearly independent rows or columns it has. For example, a 2x2 matrix where one row is just a multiple of the other has a rank of 1 (a low rank), while a 2x2 matrix with two distinct rows (not multiples of each other) would have a rank of 2 (a full rank).
*   **Low Rank:** A matrix is considered "low-rank" if its rank is significantly smaller than the maximum possible rank (which is the minimum of its number of rows and columns). Such matrices are often used to approximate higher-rank matrices, as they can represent complex data using fewer underlying dimensions.

In the context of LoRA, the idea is that the *change* (ΔW) needed for a pre-trained weight matrix `W` (e.g., from an `nn.Linear` layer) to adapt to a new task is often low-rank. Instead of directly learning the entire `ΔW` matrix, which could be very large, LoRA decomposes `ΔW` into the product of two much smaller (low-rank) matrices, `B` and `A`:

**`ΔW = B @ A`**

Where:

*   **`A`** is a matrix of shape `(r, in_features)`. In your code, `self.A` is initialized with `(r, in_f)`.
*   **`B`** is a matrix of shape `(out_features, r)`. In your code, `self.B` is initialized with `(out_f, r)`.

Here, `r` is the **rank** parameter (e.g., `r=4` in your code), which is typically much smaller than `in_features` or `out_features`. Because `r` is small, `A` and `B` are "skinny" matrices, meaning they have many fewer parameters than the original `ΔW` matrix would have had. When you multiply `B @ A`, the resulting `ΔW` matrix has a rank of at most `r`.

**Why use low-rank matrices `A` and `B`?**

1.  **Parameter Efficiency:** Instead of training `in_features * out_features` parameters for `ΔW`, you only train `r * in_features + out_features * r` parameters for `A` and `B`. This significantly reduces the number of trainable parameters.
2.  **Computational Efficiency:** Training and updating these smaller matrices is much faster.
3.  **Empirical Effectiveness:** Research suggests that the *change* a task needs for a pre-trained weight matrix is often inherently low-rank. This means that a small `r` can effectively capture the necessary adjustments without having to modify the entire original weight matrix.

## 2 · Helpers + pretrain on Task A
Same toy setup as Case 01 so you can compare directly.

In [ ]:
def make_model():
    return nn.Sequential(nn.Linear(1,64), nn.Tanh(), nn.Linear(64,64), nn.Tanh(), nn.Linear(64,1))
def task_data(phase, n=256):
    x = torch.linspace(-math.pi, math.pi, n).unsqueeze(1); return x, torch.sin(x+phase)
def train(model, x, y, steps, lr):
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    lossf = nn.MSELoss(); hist=[]
    for _ in range(steps):
        opt.zero_grad(); l=lossf(model(x),y); l.backward(); opt.step(); hist.append(l.item())
    return hist
def trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

xa,ya = task_data(0.0); model = make_model(); train(model, xa, ya, 1500, 1e-2)
print('pretrained. total params:', sum(p.numel() for p in model.parameters()))

## 3 · Inject LoRA and compare to full fine-tuning
New Task B arrives. Approach 1: full fine-tune (train everything). Approach 2: LoRA (train only adapters).

In [ ]:
def inject_lora(model, r=4, alpha=8):
    return nn.Sequential(*[LoRALinear(m,r,alpha) if isinstance(m,nn.Linear) else m for m in model])

xb,yb = task_data(1.2)
full = copy.deepcopy(model);              h_full = train(full, xb, yb, 400, 1e-3)
lora = inject_lora(copy.deepcopy(model)); h_lora = train(lora, xb, yb, 400, 1e-2)

print(f'FULL  fine-tune: {trainable(full):5d} trainable params -> loss {h_full[-1]:.5f}')
print(f'LoRA  fine-tune: {trainable(lora):5d} trainable params -> loss {h_lora[-1]:.5f}')
print(f'LoRA trains {100*trainable(lora)/trainable(full):.1f}% of the params')

In [ ]:
plt.figure(figsize=(6,3))
plt.plot(h_full, label=f'full  ({trainable(full)} params)')
plt.plot(h_lora, label=f'LoRA  ({trainable(lora)} params)')
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('loss (log)'); plt.legend()
plt.title('LoRA matches full fine-tuning with far fewer trainable params'); plt.show()

## 4 · Rank sweep — how small can r go?
Lower r = cheaper = fewer params. Find the rank where quality stops improving — that's your sweet spot.

In [ ]:
ranks = [1,2,4,8,16]; finals=[]; params=[]
for r in ranks:
    m = inject_lora(copy.deepcopy(model), r=r, alpha=2*r)
    h = train(m, xb, yb, 400, 1e-2); finals.append(h[-1]); params.append(trainable(m))
fig,ax=plt.subplots(1,2,figsize=(9,3))
ax[0].plot(ranks, finals, 'o-'); ax[0].set_xlabel('rank r'); ax[0].set_ylabel('final loss'); ax[0].set_title('quality vs rank')
ax[1].plot(ranks, params, 's-', color='green'); ax[1].set_xlabel('rank r'); ax[1].set_ylabel('trainable params'); ax[1].set_title('cost vs rank')
plt.tight_layout(); plt.show()
for r,f,p in zip(ranks,finals,params): print(f'r={r:2d}  loss={f:.5f}  params={p}')

## 5 · The base is untouched — swap adapters per task
Because W is frozen, you can keep **one small adapter per task** and load whichever you need.
This is the first real weapon against catastrophic forgetting (Case 06): the shared knowledge in W is
never overwritten.

In [ ]:
print('Base weight unchanged by LoRA training?',
      torch.equal(lora[0].base.weight, model[0].weight))

# adapter is tiny -> cheap to store many of them
adapter_state = {k:v for k,v in lora.state_dict().items() if ('A' in k or 'B' in k)}
print('adapter tensors you would save per task:', list(adapter_state.keys()))

## 6 · Merge for free inference
At deploy time you can fold the adapter back into W so there's **zero** extra inference cost.

In [ ]:
# W_merged = W + (B @ A) * scale  -> a normal Linear again
lin0 = lora[0]
with torch.no_grad():
    merged_W = lin0.base.weight + (lin0.B @ lin0.A) * lin0.scale
print('merged weight shape:', tuple(merged_W.shape), '-> identical forward, no adapter needed at inference')

## 7 · Takeaways → Case 03
1. LoRA = freeze W, learn low-rank **B·A**. Tiny trainable footprint, near-equal quality.
2. **B=0 init** → safe start. **Merge** → free inference.
3. Frozen base + per-task adapters → a path around forgetting (Case 06).

Next: **Case 03** swaps our toy `LoRALinear` for the real `peft` library and applies **QLoRA**
(LoRA on a 4-bit quantized model) to an actual Llama — the practical workflow you'll reuse for the capstone.

➡️ Do `challenge.md`, then `cases/03_qlora_real_llm/` (built next).